# Project 26 — Milestone 2

- Reads HPatches image sequences and their ground-truth homographies.
- Benchmarks **SIFT** and **ORB** with cross-checked matching and RANSAC.
- Measures matching precision, RANSAC inliers, corner error, registration success, and runtime.

**Important:** HPatches sequences contain images `1.ppm` through `6.ppm`. We evaluate image `1` against images `2`–`6` in each selected sequence, using `H_1_2` ... `H_1_6`.


## 1. Setup and Google Drive


In [1]:
from pathlib import Path
import sys, time, json, zipfile, shutil
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from IPython.display import display

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')

PROJECT = (Path('/content/drive/MyDrive/DIP Project/Compact_Feature_Matching')
           if IN_COLAB else Path.cwd() / 'Compact_Feature_Matching')
DATA = PROJECT / 'data'
OUT = PROJECT / 'milestone2' / 'student_a'
FIGURES = OUT / 'figures'
TABLES = OUT / 'tables'
for folder in (OUT, FIGURES, TABLES):
    folder.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)
cv2.setRNGSeed(SEED)
cv2.setNumThreads(1)

print('Project:', PROJECT)
print('OpenCV:', cv2.__version__)
print('Output:', OUT)

Mounted at /content/drive
Project: /content/drive/MyDrive/DIP Project/Compact_Feature_Matching
OpenCV: 5.0.0
Output: /content/drive/MyDrive/DIP Project/Compact_Feature_Matching/milestone2/student_a


## 2. Locate the HPatches sequences
This cell detects the folder that directly contains `i_*` and `v_*` sequences. If only the ZIP exists, it extracts it under `data/`. No dataset is downloaded automatically.


In [2]:
def find_sequence_root(data_folder):
    if not data_folder.exists():
        return None
    candidates = [data_folder] + [p for p in data_folder.rglob('*') if p.is_dir()]
    for p in candidates:
        if any(q.is_dir() and q.name.startswith(('i_', 'v_')) and (q / '1.ppm').exists()
               for q in p.iterdir()):
            return p
    return None

SEQ_ROOT = find_sequence_root(DATA)
if SEQ_ROOT is None:
    zip_candidates = list(DATA.rglob('hpatches-sequences-release.zip')) if DATA.exists() else []
    if not zip_candidates:
        raise FileNotFoundError(f'HPatches sequences not found in {DATA}. Place extracted sequences or the ZIP there.')
    target = DATA / 'hpatches_extracted'
    target.mkdir(parents=True, exist_ok=True)
    print('Extracting:', zip_candidates[0])
    with zipfile.ZipFile(zip_candidates[0]) as z:
        # Extract only entries that stay within the target directory
        for member in z.infolist():
            dest = (target / member.filename).resolve()
            if not dest.is_relative_to(target.resolve()):
                raise ValueError('Unsafe ZIP member path')
        z.extractall(target)
    SEQ_ROOT = find_sequence_root(target)

if SEQ_ROOT is None:
    raise FileNotFoundError('Could not locate HPatches folders with 1.ppm images.')

sequences = sorted(p for p in SEQ_ROOT.iterdir()
                   if p.is_dir() and p.name.startswith(('i_', 'v_')) and (p / '1.ppm').exists())
print('Sequence root:', SEQ_ROOT)
print('Sequences:', len(sequences))
print('Illumination:', sum(p.name.startswith('i_') for p in sequences))
print('Viewpoint:', sum(p.name.startswith('v_') for p in sequences))
assert sequences, 'No sequences found'

Extracting: /content/drive/MyDrive/DIP Project/Compact_Feature_Matching/data/hpatches-sequences-release.zip
Sequence root: /content/drive/MyDrive/DIP Project/Compact_Feature_Matching/data/hpatches_extracted/hpatches-sequences-release
Sequences: 116
Illumination: 57
Viewpoint: 59


## 3. Evaluation configuration
`MAX_SEQUENCES=None` evaluates all available sequences (typically 116). Set an integer such as `20` only if you need a **documented subset**. The first `N` sequences are selected alphabetically; no random sampling is implied.

`MATCH_THRESHOLD_PX=3` is used to judge matches using ground-truth homography. RANSAC uses the same 3-pixel reprojection threshold. The reported precision is the fraction of all tentative matches with ground-truth error below 3 px.


In [3]:
MAX_SEQUENCES = None
MATCH_THRESHOLD_PX = 3.0
CORNER_SUCCESS_THRESHOLDS = (3.0, 5.0)
NFEATURES = 1000
RANSAC_THRESHOLD_PX = 3.0

selected_sequences = sequences if MAX_SEQUENCES is None else sequences[:MAX_SEQUENCES]
pairs = []
missing = []
for seq in selected_sequences:
    for image_id in range(2, 7):
        im1, im2, hfile = seq / '1.ppm', seq / f'{image_id}.ppm', seq / f'H_1_{image_id}'
        if im1.exists() and im2.exists() and hfile.exists():
            pairs.append((seq, image_id))
        else:
            missing.append((seq.name, image_id))
print('Selected sequences:', len(selected_sequences))
print('Valid image pairs:', len(pairs))
print('Missing pairs:', len(missing))
assert pairs, 'No valid evaluation pairs'

Selected sequences: 116
Valid image pairs: 580
Missing pairs: 0


## 4. Baseline functions
Each method performs **feature extraction → BFMatcher → ground-truth match verification → RANSAC homography → corner error**.

- **SIFT:** 128-dimensional float descriptors; Euclidean (`L2`) distance.
- **ORB:** 32-byte binary descriptors; Hamming distance.
- **RANSAC inliers** are algorithm-selected; **ground-truth correct matches** are independently defined using the known `H_1_j`.
- **Corner error:** mean Euclidean distance between the four corners projected by the estimated and ground-truth homographies.
- Timing is per pair, in milliseconds, using `time.perf_counter()`. It excludes image disk loading and plotting. It includes both images' extraction, matching and RANSAC geometry.


In [4]:
def project_points(points, H):
    points = np.asarray(points, dtype=np.float64).reshape(-1, 1, 2)
    projected = cv2.perspectiveTransform(points, H).reshape(-1, 2)
    return projected

def corner_error(H_gt, H_est, image_shape):
    if H_est is None:
        return np.nan
    h, w = image_shape[:2]
    corners = np.array([[0, 0], [w-1, 0], [w-1, h-1], [0, h-1]], dtype=np.float64)
    gt = project_points(corners, H_gt)
    est = project_points(corners, H_est)
    if not np.isfinite(gt).all() or not np.isfinite(est).all():
        return np.nan
    return float(np.linalg.norm(gt - est, axis=1).mean())

def benchmark_pair(img1, img2, H_gt, method):
    if method == 'SIFT':
        extractor = cv2.SIFT_create(nfeatures=NFEATURES)
        norm = cv2.NORM_L2
    elif method == 'ORB':
        extractor = cv2.ORB_create(nfeatures=NFEATURES)
        norm = cv2.NORM_HAMMING
    else:
        raise ValueError(method)

    t0 = time.perf_counter()
    kp1, des1 = extractor.detectAndCompute(img1, None)
    kp2, des2 = extractor.detectAndCompute(img2, None)
    t1 = time.perf_counter()

    matches = []
    if des1 is not None and des2 is not None:
        matcher = cv2.BFMatcher(norm, crossCheck=True)
        matches = sorted(matcher.match(des1, des2), key=lambda m: m.distance)
    t2 = time.perf_counter()

    points1 = np.array([kp1[m.queryIdx].pt for m in matches], dtype=np.float64).reshape(-1, 2)
    points2 = np.array([kp2[m.trainIdx].pt for m in matches], dtype=np.float64).reshape(-1, 2)
    correct = np.zeros(len(matches), dtype=bool)
    if matches:
        expected = project_points(points1, H_gt)
        errors = np.linalg.norm(expected - points2, axis=1)
        correct = np.isfinite(errors) & (errors < MATCH_THRESHOLD_PX)

    H_est, ransac_mask = None, None
    if len(matches) >= 4:
        cv2.setRNGSeed(SEED)
        try:
            H_est, ransac_mask = cv2.findHomography(
                points1, points2, cv2.RANSAC,
                ransacReprojThreshold=RANSAC_THRESHOLD_PX
            )
        except cv2.error:
            H_est, ransac_mask = None, None
    t3 = time.perf_counter()

    inlier_mask = (ransac_mask.ravel().astype(bool)
                   if ransac_mask is not None else np.zeros(len(matches), dtype=bool))
    inliers = int(inlier_mask.sum())
    correct_inliers = int((inlier_mask & correct).sum())
    c_err = corner_error(H_gt, H_est, img1.shape)
    result = {
        'method': method, 'keypoints_1': len(kp1), 'keypoints_2': len(kp2),
        'matches': len(matches), 'correct_matches': int(correct.sum()),
        'incorrect_matches': int(len(matches)-correct.sum()),
        'matching_precision': float(correct.mean()) if matches else np.nan,
        'ransac_inliers': inliers,
        'ransac_outliers': len(matches)-inliers,
        'ransac_correct_inliers': correct_inliers,
        'precision_after_ransac': correct_inliers/inliers if inliers else np.nan,
        'corner_error_px': c_err,
        'success_at_3px': bool(np.isfinite(c_err) and c_err < 3.0),
        'success_at_5px': bool(np.isfinite(c_err) and c_err < 5.0),
        'homography_found': bool(H_est is not None),
        'extraction_ms': (t1-t0)*1000,
        'matching_ms': (t2-t1)*1000,
        'geometry_ms': (t3-t2)*1000,
        'total_ms': (t3-t0)*1000
    }
    visualization = {'kp1': kp1, 'kp2': kp2, 'matches': matches, 'H_est': H_est}
    return result, visualization

## 5. Run the full baseline benchmark
 Each sequence contributes up to five image pairs. Both methods are tested on **exactly the same pairs**.


In [5]:
records = []
example_visualizations = {}
started = time.perf_counter()

for idx, (seq, image_id) in enumerate(pairs, start=1):
    img1 = cv2.imread(str(seq / '1.ppm'), cv2.IMREAD_GRAYSCALE)
    img2 = cv2.imread(str(seq / f'{image_id}.ppm'), cv2.IMREAD_GRAYSCALE)
    if img1 is None or img2 is None:
        raise FileNotFoundError(f'Failed to read images in {seq.name}, pair 1-{image_id}')
    H_gt = np.loadtxt(seq / f'H_1_{image_id}').reshape(3, 3)
    if not np.isfinite(H_gt).all():
        raise ValueError(f'Invalid homography: {seq.name} H_1_{image_id}')

    for method in ('SIFT', 'ORB'):
        result, vis = benchmark_pair(img1, img2, H_gt, method)
        result.update({'sequence': seq.name, 'category': 'illumination' if seq.name.startswith('i_') else 'viewpoint',
                       'target_image': image_id})
        records.append(result)
        if (seq.name, image_id) == (pairs[0][0].name, pairs[0][1]):
            example_visualizations[method] = (img1.copy(), img2.copy(), vis)

    if idx % 25 == 0 or idx == len(pairs):
        print(f'Processed {idx}/{len(pairs)} pairs')

results = pd.DataFrame(records)
print(f'Completed in {(time.perf_counter()-started)/60:.2f} minutes')
display(results.head())
results.to_csv(TABLES / 'm2_student_a_pair_results.csv', index=False)

Processed 25/580 pairs
Processed 50/580 pairs
Processed 75/580 pairs
Processed 100/580 pairs
Processed 125/580 pairs
Processed 150/580 pairs
Processed 175/580 pairs
Processed 200/580 pairs
Processed 225/580 pairs
Processed 250/580 pairs
Processed 275/580 pairs
Processed 300/580 pairs
Processed 325/580 pairs
Processed 350/580 pairs
Processed 375/580 pairs
Processed 400/580 pairs
Processed 425/580 pairs
Processed 450/580 pairs
Processed 475/580 pairs
Processed 500/580 pairs
Processed 525/580 pairs
Processed 550/580 pairs
Processed 575/580 pairs
Processed 580/580 pairs
Completed in 8.81 minutes


,method,keypoints_1,keypoints_2,matches,correct_matches,incorrect_matches,matching_precision,ransac_inliers,ransac_outliers,ransac_correct_inliers,...,success_at_3px,success_at_5px,homography_found,extraction_ms,matching_ms,geometry_ms,total_ms,sequence,category,target_image
0,SIFT,840,780,417,324,93,0.776978,317,100,317,...,False,True,True,466.973946,81.237670,16.111911,564.323527,i_ajuntament,illumination,2
1,ORB,1000,1000,585,520,65,0.888889,518,67,516,...,True,True,True,31.914613,57.097058,3.337538,92.349209,i_ajuntament,illumination,2
2,SIFT,840,417,164,25,139,0.152439,15,149,14,...,False,False,True,370.612413,45.820748,77.887384,494.320545,i_ajuntament,illumination,3
3,ORB,1000,1000,222,52,170,0.234234,52,170,50,...,False,False,True,26.036255,57.645945,84.694587,168.376787,i_ajuntament,illumination,3
4,SIFT,840,302,125,11,114,0.088000,7,118,0,...,False,False,True,339.439753,30.343234,72.344004,442.126991,i_ajuntament,illumination,4


## 6. Full baseline tables
The main summary uses **macro averages across image pairs**. A missing homography counts as a registration failure, but its corner error is `NaN` and excluded from mean corner error. Precision is undefined for pairs with zero matches.

We also save a breakdown for illumination (`i_*`) versus viewpoint (`v_*`) sequences.


In [6]:
def make_summary(df, group_cols):
    summary = df.groupby(group_cols, dropna=False).agg(
        image_pairs=('matches', 'size'),
        mean_keypoints_1=('keypoints_1', 'mean'),
        mean_keypoints_2=('keypoints_2', 'mean'),
        mean_matches=('matches', 'mean'),
        mean_correct_matches=('correct_matches', 'mean'),
        mean_precision=('matching_precision', 'mean'),
        mean_ransac_inliers=('ransac_inliers', 'mean'),
        mean_precision_after_ransac=('precision_after_ransac', 'mean'),
        mean_corner_error_px=('corner_error_px', 'mean'),
        registration_success_3px=('success_at_3px', 'mean'),
        registration_success_5px=('success_at_5px', 'mean'),
        homography_success_rate=('homography_found', 'mean'),
        extraction_ms=('extraction_ms', 'mean'),
        matching_ms=('matching_ms', 'mean'),
        geometry_ms=('geometry_ms', 'mean'),
        total_ms=('total_ms', 'mean')
    ).reset_index()
    return summary

summary = make_summary(results, ['method'])
by_category = make_summary(results, ['category', 'method'])
summary.to_csv(TABLES / 'm2_student_a_baseline_summary.csv', index=False)
by_category.to_csv(TABLES / 'm2_student_a_by_category.csv', index=False)

display(summary.round(3))
display(by_category.round(3))

,method,image_pairs,mean_keypoints_1,mean_keypoints_2,mean_matches,mean_correct_matches,mean_precision,mean_ransac_inliers,mean_precision_after_ransac,mean_corner_error_px,registration_success_3px,registration_success_5px,homography_success_rate,extraction_ms,matching_ms,geometry_ms,total_ms
0,ORB,580,992.931,991.700,369.819,204.333,0.473,209.247,0.848,216.600,0.490,0.629,1.0,49.800,36.414,20.891,107.105
1,SIFT,580,965.224,947.779,388.790,220.874,0.497,224.712,0.892,120.717,0.695,0.793,1.0,682.925,75.719,16.963,775.607


,category,method,image_pairs,mean_keypoints_1,mean_keypoints_2,mean_matches,mean_correct_matches,mean_precision,mean_ransac_inliers,mean_precision_after_ransac,mean_corner_error_px,registration_success_3px,registration_success_5px,homography_success_rate,extraction_ms,matching_ms,geometry_ms,total_ms
0,illumination,ORB,285,989.982,988.070,374.095,208.568,0.462,216.456,0.844,155.977,0.621,0.730,1.0,42.337,37.448,22.210,101.994
1,illumination,SIFT,285,958.895,924.993,379.625,210.316,0.480,218.681,0.894,75.589,0.811,0.877,1.0,620.842,75.223,18.161,714.225
2,viewpoint,ORB,295,995.780,995.207,365.688,200.241,0.485,202.281,0.852,275.168,0.363,0.532,1.0,57.011,35.415,19.616,112.042
3,viewpoint,SIFT,295,971.339,969.793,397.644,231.075,0.513,230.539,0.889,164.316,0.583,0.712,1.0,742.904,76.198,15.806,834.907


## 7. Save report-ready tables as individual PNG files
Each table is also available as CSV.

In [7]:
def save_table_png(df, path, title, decimals=2):
    view = df.copy()
    for col in view.select_dtypes(include='number').columns:
        view[col] = view[col].map(lambda x: f'{x:.{decimals}f}' if pd.notna(x) else 'N/A')
    fig_width = max(9, len(view.columns)*1.35)
    fig_height = max(2.3, 0.6*len(view)+1.2)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis('off')
    ax.set_title(title, fontsize=14, weight='bold', pad=18)
    table = ax.table(cellText=view.values, colLabels=view.columns,
                     cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.5)
    fig.savefig(path, dpi=180, bbox_inches='tight')
    plt.close(fig)

report_cols = ['method', 'image_pairs', 'mean_precision',
               'mean_precision_after_ransac', 'mean_corner_error_px',
               'registration_success_3px', 'total_ms']
save_table_png(summary[report_cols], FIGURES / 'm2_baseline_summary_table.png',
               'Milestone 2 — SIFT vs ORB')
save_table_png(by_category[['category']+report_cols], FIGURES / 'm2_category_table.png',
               'Illumination vs Viewpoint Baselines')
print('Saved separate table figures.')

Saved separate table figures.


## 8. Save individual accuracy, registration and timing plots

In [8]:
def save_bar(metric, ylabel, filename, percent=False):
    fig, ax = plt.subplots(figsize=(6, 4))
    vals = summary.set_index('method').reindex(['SIFT', 'ORB'])[metric]
    ax.bar(vals.index, vals.values)
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel + ' — SIFT vs ORB')
    if percent:
        ax.set_ylim(0, 1)
        ax.set_yticks(np.linspace(0, 1, 6))
        ax.set_yticklabels([f'{int(x*100)}%' for x in np.linspace(0, 1, 6)])
    fig.tight_layout()
    fig.savefig(FIGURES / filename, dpi=180)
    plt.close(fig)

save_bar('mean_precision', 'Matching precision', 'm2_matching_precision.png', True)
save_bar('mean_precision_after_ransac', 'Precision after RANSAC', 'm2_ransac_precision.png', True)
save_bar('mean_corner_error_px', 'Mean corner error (px)', 'm2_corner_error.png')
save_bar('registration_success_3px', 'Registration success @ 3 px', 'm2_registration_success.png', True)
save_bar('total_ms', 'Mean total runtime (ms)', 'm2_total_runtime.png')

timing_cols = ['extraction_ms', 'matching_ms', 'geometry_ms']
timing = summary.set_index('method').reindex(['SIFT', 'ORB'])
fig, ax = plt.subplots(figsize=(7, 4))
bottom = np.zeros(len(timing))
for col in timing_cols:
    vals = timing[col].to_numpy()
    ax.bar(timing.index, vals, bottom=bottom, label=col.replace('_ms', ''))
    bottom += vals
ax.set_ylabel('Mean runtime (ms)')
ax.set_title('Runtime breakdown')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / 'm2_timing_breakdown.png', dpi=180)
plt.close(fig)
print('Saved six separate plots.')

Saved six separate plots.


## 9. Save example match visualizations
The images are only qualitative examples; the numerical tables above use all selected pairs.

In [9]:
for method, (img1, img2, vis) in example_visualizations.items():
    matched = cv2.drawMatches(
        img1, vis['kp1'], img2, vis['kp2'],
        vis['matches'][:40], None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )
    path = FIGURES / f'm2_{method.lower()}_matches.png'
    cv2.imwrite(str(path), matched)
    print(path.name)

print('\nFigures:')
for p in sorted(FIGURES.iterdir()):
    print(' ', p.name)

m2_sift_matches.png
m2_orb_matches.png

Figures:
  m2_baseline_summary_table.png
  m2_category_table.png
  m2_corner_error.png
  m2_matching_precision.png
  m2_orb_matches.png
  m2_ransac_precision.png
  m2_registration_success.png
  m2_sift_matches.png
  m2_timing_breakdown.png
  m2_total_runtime.png


## 10. Save reproducibility metadata and export ZIP
The ZIP contains all separate figures and CSV files. Dataset files and neural-network checkpoints are not included.

In [12]:
metadata = {
    'project': 'Project 26',
    'milestone': 2,
    'sequence_root': str(SEQ_ROOT),
    'selected_sequences': [p.name for p in selected_sequences],
    'number_of_image_pairs': len(pairs),
    'methods': ['SIFT', 'ORB'],
    'nfeatures': NFEATURES,
    'matching': 'BFMatcher crossCheck=True',
    'match_distance': {'SIFT': 'L2', 'ORB': 'Hamming'},
    'matching_precision_threshold_px': MATCH_THRESHOLD_PX,
    'ransac_threshold_px': RANSAC_THRESHOLD_PX,
    'random_seed': SEED,
    'opencv_version': cv2.__version__,
    'timing_excludes': ['image loading', 'visualization', 'CSV writing']
}
(OUT / 'benchmark_config.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')

archive_path = shutil.make_archive(str(OUT / 'student_a_m2_exports'), 'zip', root_dir=OUT,
                                   base_dir='figures')
# CSV files are separately available in tables/; create a complete archive as well.
import zipfile
complete_zip = OUT / 'student_a_m2_complete_results.zip'
with zipfile.ZipFile(complete_zip, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for folder in (FIGURES, TABLES):
        for p in folder.iterdir():
            if p.is_file():
                z.write(p, arcname=f'{folder.name}/{p.name}')
    z.write(OUT / 'benchmark_config.json', arcname='benchmark_config.json')
print('All results:', OUT)
print('Complete ZIP:', complete_zip)
if IN_COLAB:
    from google.colab import files
    # Uncomment if you want the ZIP downloaded to your computer:
    # files.download(str(complete_zip))

All results: /content/drive/MyDrive/DIP Project/Compact_Feature_Matching/milestone2/student_a
Complete ZIP: /content/drive/MyDrive/DIP Project/Compact_Feature_Matching/milestone2/student_a/student_a_m2_complete_results.zip


## Interpretation
1. Report the **exact sequence and image-pair counts** (and any missing pairs).
2. Compare **matching precision, mean corner error, registration success, and total runtime**. Higher precision is better; lower error and runtime are better.
3. A method can have many RANSAC inliers but still produce an inaccurate homography; therefore report **both** inliers and corner error.

